## Purpose
This notebook will identify last 3 months best perfomer

In [72]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_date, date_sub
import configparser
import os
import pandas as pd
import matplotlib.pyplot as plt


In [73]:
config = configparser.ConfigParser()
config.read(os.path.join(os.path.dirname(os.path.abspath("__file__")), '../conf/config.ini'))
url = 'jdbc:mysql://localhost/{}'.format(config.get('mysql', 'database'))
driver = config.get('mysql', 'driver')
tablename = "stocksinfp"
username = config.get('mysql', 'username')
password = config.get('mysql', 'password')
print("url : ", url)
print("Driver : ", driver)
print("MySQL User : ", username)

url :  jdbc:mysql://localhost/stocksdb
Driver :  com.mysql.cj.jdbc.Driver
MySQL User :  quizadmin


In [74]:
spark = SparkSession.builder.appName("Best25Stocks") \
.master("local[*]") \
.config("spark.jars",
                "/Users/gaurav/.m2/repository/com/mysql/mysql-connector-j/8.0.33/mysql-connector-j-8.0.33.jar") \
.getOrCreate()

In [75]:
stock_df = spark.read.format("jdbc").options(url=url, driver=driver, user=username, password=password, dbtable=tablename).load()
#spark.sparkContext.stop()

In [76]:
# Filter for records in the last 1 year
one_year_ago = date_sub(current_date(), 180)
filtered_df = stock_df.filter(col("Date") >= one_year_ago)
#filtered_df = filtered_df.filter(col("Symbol") == 'AAPL')
# Show the result
filtered_df.printSchema()
filtered_df.show(2, truncate=False)

root
 |-- Date: date (nullable = true)
 |-- Open: string (nullable = true)
 |-- High: string (nullable = true)
 |-- Low: string (nullable = true)
 |-- Close: string (nullable = true)
 |-- Adj Close: string (nullable = true)
 |-- Volume: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- price_change: double (nullable = true)

+----------+------------------+------------------+------------------+------------------+------------------+--------+------+------------------+
|Date      |Open              |High              |Low               |Close             |Adj Close         |Volume  |symbol|price_change      |
+----------+------------------+------------------+------------------+------------------+------------------+--------+------+------------------+
|2024-08-14|220.57000732421875|223.02999877929688|219.6999969482422 |221.72000122070312|221.72000122070312|41960600|AAPL  |0.4499969482421875|
|2024-08-14|191.0             |193.35000610351562|190.47000122070312|193.0         

In [77]:
import pandas as pd
import numpy as np

def analyze_stock_data(df: pd.DataFrame):
    """
    Analyze stock market data and calculate key technical indicators
    
    Parameters:
    df (pandas.DataFrame): DataFrame with columns Date, Open, High, Low, Close, Volume, symbol
    
    Returns:
    pandas.DataFrame: Original data with additional technical indicators
    dict: Summary statistics
    """
    # Create a copy using pandas method
    analysis = df.copy(deep=True)
    
    # Convert string columns to numeric
    numeric_columns = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    for col in numeric_columns:
        analysis[col] = pd.to_numeric(analysis[col], errors='coerce')
    
    # Calculate daily trading range
    analysis['daily_range'] = analysis['High'] - analysis['Low']
    
    # Calculate moving averages
    analysis['MA_20'] = analysis['Close'].rolling(window=20).mean()
    analysis['MA_50'] = analysis['Close'].rolling(window=50).mean()
    
    # Calculate daily volatility (standard deviation of price changes)
    analysis['volatility'] = analysis['price_change'].rolling(window=20).std()
    
    # Calculate trading volume moving average
    analysis['volume_MA_20'] = analysis['Volume'].rolling(window=20).mean()
    
    # Calculate relative strength (ratio of current price to 20-day moving average)
    analysis['relative_strength'] = analysis['Close'] / analysis['MA_20']
    
    # Generate summary statistics
    summary = {
        'latest_price': float(analysis['Close'].iloc[-1]),
        'avg_daily_volume': float(analysis['Volume'].mean()),
        'avg_daily_range': float(analysis['daily_range'].mean()),
        'max_price': float(analysis['High'].max()),
        'min_price': float(analysis['Low'].min()),
        'avg_price_change': float(analysis['price_change'].mean()),
        'volatility': float(analysis['price_change'].std()),
        'total_trading_days': len(analysis),
        'date_range': f"{analysis['Date'].min()} to {analysis['Date'].max()}"
    }
    
    return analysis, summary


In [83]:
def calculate_technical_indicators(df: pd.DataFrame):
    """
    Calculate additional technical indicators for stock analysis
    
    Parameters:
    df (pandas.DataFrame): DataFrame with stock data
    
    Returns:
    pandas.DataFrame: DataFrame with additional technical indicators
    """
    data = df.copy(deep=True)
    
    # Calculate RSI (Relative Strength Index)
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    data['RSI'] = 100 - (100 / (1 + rs))
    
    # Calculate MACD (Moving Average Convergence Divergence)
    exp1 = data['Close'].ewm(span=12, adjust=False).mean()
    exp2 = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD'] = exp1 - exp2
    data['Signal_Line'] = data['MACD'].ewm(span=9, adjust=False).mean()
    
    # Calculate Bollinger Bands
    data['BB_middle'] = data['Close'].rolling(window=20).mean()
    bb_std = data['Close'].rolling(window=20).std()
    data['BB_upper'] = data['BB_middle'] + (bb_std * 2)
    data['BB_lower'] = data['BB_middle'] - (bb_std * 2)
    
    return data

In [84]:
# First analyze basic metrics
analyzed_df, summary = analyze_stock_data(filtered_df)

# Then add technical indicators
final_df = calculate_technical_indicators(analyzed_df)

AttributeError: 'DataFrame' object has no attribute 'copy'